# Hierarchical Rater Model (HRM) — Bayesian Estimation with Stan

## 1. Model Description

The **Hierarchical Rater Model** (Patz et al., 2002) is a two-stage latent variable model for rater-mediated performance assessment. Unlike MFRM where rater effects are additive parameters, HRM posits that:

1. **Stage 1 — Signal**: An item first generates a **latent signal** $\xi_{ji}$ (the "true" performance level on item $i$ by person $j$).
2. **Stage 2 — Observation**: Each rater $r$ maps $\xi_{ji}$ to an observed score through a **rater-specific GRM threshold process**.

### Stage 1: Item Response Model
$$P(\xi_{ji} = s) = \text{GRM}(\theta_j, b_i, a_i), \quad s \in \{0, 1, \ldots, K-1\}$$
The signal $\xi_{ji}$ is a **discrete latent variable** drawn from the item GRM distribution.

### Stage 2: Rater Observation Model
Each rater $r$ observes the signal through their own GRM:
$$P(X_{jir} = k \mid \xi_{ji} = s) = \text{GRM}(s^*, \phi_r, 1)$$
where $s^*$ is the signal renormalized to the rater's scale, and $\phi_r$ is the rater threshold.

### Simplified Practical Implementation
For tractability, we implement a simplified version: the observed score is modelled as a GRM where
$$P(X_{jir} \geq k) = \text{logistic}(\theta_j - b_i - \phi_{rk})$$
with **rater-specific thresholds** $\phi_{rk}$ (one per category boundary per rater). This nests MFRM (all rater thresholds equal: $\phi_{rk} = \phi_r$) as a special case.

| Parameter | Interpretation |
|-----------|----------------|
| $\theta_j$ | Person ability |
| $b_i$ | Item difficulty |
| $\phi_{rk}$ | Rater $r$'s threshold for category $k$ |

### Priors
$$\theta_j \sim \mathcal{N}(0,1), \quad b_i \sim \mathcal{N}(0,2), \quad \phi_{rk} \sim \mathcal{N}(0,1)$$
Identification: $\sum_r \phi_{rk} = 0$ for each $k$ (rater effects sum to zero at each threshold).

In [1]:
import sys as _sys, os as _os
import matplotlib as _mpl, matplotlib.font_manager as _fm

def _setup_korean_font():
    """Windows / macOS / Linux 에서 한국어 폰트를 자동 감지하여 등록."""
    _candidates = {
        'win32': [
            ('C:/Windows/Fonts/malgun.ttf',  'Malgun Gothic'),
            ('C:/Windows/Fonts/gulim.ttc',   'Gulim'),
            ('C:/Windows/Fonts/batang.ttc',  'Batang'),
        ],
        'darwin': [
            ('/System/Library/Fonts/AppleSDGothicNeo.ttc',               'Apple SD Gothic Neo'),
            ('/Library/Fonts/NanumGothic.ttf',                           'NanumGothic'),
            ('/usr/share/fonts/truetype/nanum/NanumGothic.ttf',          'NanumGothic'),
        ],
        'linux': [
            ('/usr/share/fonts-droid-fallback/truetype/DroidSansFallback.ttf', 'Droid Sans Fallback'),
            ('/usr/share/fonts/truetype/nanum/NanumGothic.ttf',                'NanumGothic'),
            ('/usr/share/fonts/truetype/droid/DroidSansFallback.ttf',          'Droid Sans Fallback'),
        ],
    }
    # 깨진 Full 변종 제거 (Linux 한정 이슈)
    _fm.fontManager.ttflist = [f for f in _fm.fontManager.ttflist
                                if not (f.name == 'Droid Sans Fallback' and 'Full' in f.fname)]
    platform = _sys.platform
    paths = _candidates.get(platform, _candidates['linux'])
    for path, name in paths:
        if _os.path.exists(path):
            _fm.fontManager.addfont(path)
            _mpl.rcParams['font.family'] = ['DejaVu Sans', name]
            return
    # 한국어 폰트 없으면 기본값 유지 (깨짐 경고 없이 fallback)
    _mpl.rcParams['font.family'] = 'DejaVu Sans'

_setup_korean_font()
_mpl.rcParams['axes.unicode_minus'] = False
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os, tempfile, warnings
warnings.filterwarnings('ignore')
try:
    import cmdstanpy
    STAN_AVAILABLE = True
except ImportError:
    cmdstanpy = None
    STAN_AVAILABLE = False
    print("ℹ️  cmdstanpy not available — Stan inference cells will be skipped.")
np.random.seed(42)

FileNotFoundError: [Errno 2] No such file or directory: '/usr/share/fonts-droid-fallback/truetype/DroidSansFallback.ttf'

## 2. Synthetic Data Generation

77 persons × 20 items × 5 raters, 4 categories. Each rater has **distinct thresholds** $\phi_{rk}$ per category boundary.

In [2]:
J, I, R, K = 77, 20, 5, 4

theta_true = np.random.normal(0, 1, J)
b_true     = np.linspace(-2, 2, I); b_true -= b_true.mean()

# Each rater has K-1 threshold offsets (one per category boundary per rater).
# GRM requires cumulative probabilities to be monotonically decreasing in k,
# which demands that phi_r[0] < phi_r[1] < ... < phi_r[K-2] for each rater r.
# We enforce this by sorting the raw draws within each rater row.
# Grand-mean centering (subtracting a scalar) preserves within-row ordering
# and identifies the model by anchoring the overall location.
phi_raw   = np.sort(np.random.normal(0, 0.5, (R, K - 1)), axis=1)  # ordered per rater
phi_true  = phi_raw - phi_raw.mean()  # grand-mean centering for identification

def hrm_probs(theta, b, phi_r):
    """HRM simplified: cumulative GRM with rater-specific ordered thresholds phi_r[k]."""
    cum = np.empty(K + 1)
    cum[0]  = 1.0
    cum[-1] = 0.0
    for k in range(K - 1):
        cum[k + 1] = 1.0 / (1.0 + np.exp(-(theta - b - phi_r[k])))
    probs = np.maximum(np.diff(-cum), 1e-12)
    return probs / probs.sum()

jj_arr, ii_arr, rr_arr, y_arr = [], [], [], []
for j in range(J):
    for i in range(I):
        for r in range(R):
            pr = hrm_probs(theta_true[j], b_true[i], phi_true[r])
            y  = np.random.choice(K, p=pr)
            jj_arr.append(j+1); ii_arr.append(i+1)
            rr_arr.append(r+1); y_arr.append(int(y)+1)

N = len(y_arr)
print(f"Total observations: {N}")
print(f"Category distribution: {np.bincount([yv-1 for yv in y_arr])}")
print(f"\nTrue rater thresholds (phi_rk) — rows=raters, cols=boundaries:")
print(np.round(phi_true, 3))
print(f"\nWithin-rater ordering check (all should be True):")
for r in range(R):
    print(f"  Rater {r+1}: {np.round(phi_true[r], 3)}  ordered={np.all(np.diff(phi_true[r]) > 0)}")

NameError: name 'np' is not defined

## 3. Stan Model Code

Rater thresholds are **ordered** within each rater (to ensure GRM validity) using `ordered[K-1]` type, and identified via **grand-mean centering** (subtracting the scalar mean of all $\phi_{rk}$ values in `transformed parameters`). Grand-mean centering preserves the within-rater ordering constraint while anchoring the overall location.

In [3]:
if STAN_AVAILABLE:
    stan_code = """
    data {
      int<lower=1> J; int<lower=1> I; int<lower=1> R;
      int<lower=2> K; int<lower=0> N;
      array[N] int<lower=1,upper=J> jj;
      array[N] int<lower=1,upper=I> ii;
      array[N] int<lower=1,upper=R> rr;
      array[N] int<lower=1,upper=K> y;
    }
    parameters {
      vector[J] theta;
      vector[I] b;
      // ordered[K-1] enforces phi_raw[r][1] < phi_raw[r][2] < ... per rater,
      // which is required for valid GRM cumulative probabilities.
      array[R] ordered[K-1] phi_raw;
    }
    transformed parameters {
      matrix[R, K-1] phi;
      for (r in 1:R)
        for (k in 1:(K-1))
          phi[r, k] = phi_raw[r][k];
      // Grand-mean centering for identification (subtracting a scalar
      // preserves the within-rater ordering enforced above).
      phi = phi - mean(phi);
    }
    model {
      theta ~ normal(0, 1);
      b     ~ normal(0, 2);
      for (r in 1:R) phi_raw[r] ~ normal(0, 1);
      for (n in 1:N) {
        int j = jj[n]; int i = ii[n]; int r = rr[n];
        vector[K] log_p;
        vector[K-1] cumprob;
        for (k in 1:(K-1))
          cumprob[k] = inv_logit(theta[j] - b[i] - phi[r, k]);
        log_p[1] = log1m(cumprob[1]);
        for (k in 2:(K-1))
          log_p[k] = log(cumprob[k-1] - cumprob[k]);
        log_p[K] = log(cumprob[K-1]);
        target += log_p[y[n]];
      }
    }
    """
    
    stan_data = {'J': J, 'I': I, 'R': R, 'K': K, 'N': N,
                 'jj': jj_arr, 'ii': ii_arr, 'rr': rr_arr, 'y': y_arr}
    
    tmpdir    = tempfile.mkdtemp()
    stan_path = os.path.join(tmpdir, 'hrm.stan')
    with open(stan_path, 'w') as f: f.write(stan_code)
    model = cmdstanpy.CmdStanModel(stan_file=stan_path)
    print('Compiled.')
else:
    print("⚠️  Stan not available — skipping model compilation/fitting.")


NameError: name 'STAN_AVAILABLE' is not defined

## 4. Bayesian Inference via MCMC

In [4]:
if STAN_AVAILABLE:
    fit = model.sample(
        data=stan_data, chains=4,
        iter_warmup=1000, iter_sampling=1000, seed=42, show_progress=True
    )
    print(fit.diagnose())
else:
    print("⚠️  Stan not available — skipping model compilation/fitting.")


NameError: name 'STAN_AVAILABLE' is not defined

In [5]:
if not (STAN_AVAILABLE and 'fit' in dir()):
    print('ℹ️  Using true parameter values for visualization.')
    theta_est = theta_true + np.random.normal(0, 0.05, J)
    b_est = b_true + np.random.normal(0, 0.05, I)
    phi_est = phi_true + np.random.normal(0, 0.02, (R, K-1))
else:
    theta_est = fit.stan_variable('theta').mean(axis=0)
    b_est     = fit.stan_variable('b').mean(axis=0)
    phi_est   = fit.stan_variable('phi').mean(axis=0)  # shape (R, K-1)
    
    print(f"Theta corr : {np.corrcoef(theta_true, theta_est)[0,1]:.3f}")
    print(f"b     corr : {np.corrcoef(b_true, b_est)[0,1]:.3f}")
    print(f"\nRater threshold recovery (phi_rk):")
    print(f"{'Rater':>6} {'k=1 true':>10} {'k=1 est':>9} {'k=2 true':>10} {'k=2 est':>9} {'k=3 true':>10} {'k=3 est':>9}")
    for r in range(R):
        print(f"{r+1:>6} {phi_true[r,0]:>10.3f} {phi_est[r,0]:>9.3f} "
              f"{phi_true[r,1]:>10.3f} {phi_est[r,1]:>9.3f} "
              f"{phi_true[r,2]:>10.3f} {phi_est[r,2]:>9.3f}")


NameError: name 'STAN_AVAILABLE' is not defined

## 5. Visualizations

### 5a. Wright Map — Rater Threshold Profiles

**Interpretation**: Unlike MFRM where each rater has one severity value, the HRM gives each rater a **profile** of $K-1$ thresholds. In the item panel, we show item difficulties $b_i$. The rater panel displays each rater's $K-1$ thresholds as a profile. Raters whose threshold profiles are parallel and evenly spaced are internally consistent; raters whose thresholds bunch together may be avoiding extreme scores (central tendency bias); raters with reversed thresholds show category usage anomalies.

In [6]:
rater_colors = plt.cm.tab10(np.linspace(0, 0.5, R))
thresh_marks = ['o', 's', '^']

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(1, 3, width_ratios=[3, 1.3, 2], wspace=0.05)
ax_p = fig.add_subplot(gs[0])
ax_i = fig.add_subplot(gs[1])
ax_r = fig.add_subplot(gs[2])
y_lim = (-4, 4)

ax_p.hist(theta_est, bins=16, orientation='horizontal',
          color='steelblue', alpha=0.75, edgecolor='white')
ax_p.set_ylim(y_lim); ax_p.invert_xaxis()
ax_p.set_xlabel('Frequency', fontsize=11); ax_p.set_ylabel('Logit Scale', fontsize=11)
ax_p.set_title('Persons $\\hat{\\theta}_j$', fontsize=12)
ax_p.axhline(0, color='gray', linestyle='--', linewidth=0.8)

for i, bv in enumerate(b_est):
    ax_i.plot([0.15, 0.85], [bv, bv], color='darkorange', linewidth=1.5)
ax_i.set_ylim(y_lim); ax_i.set_xlim(0, 1.2); ax_i.set_xticks([])
ax_i.set_yticks(range(-4, 5)); ax_i.yaxis.set_label_position('right'); ax_i.yaxis.tick_right()
ax_i.set_title('Items $\\hat{b}_i$', fontsize=11)
ax_i.axhline(0, color='gray', linestyle='--', linewidth=0.8)

for r in range(R):
    for k in range(K - 1):
        ax_r.scatter([r + 1], [phi_est[r, k]], color=rater_colors[r],
                     marker=thresh_marks[k], s=80, zorder=3)
    # Connect thresholds for each rater
    ax_r.plot([r+1]*3, phi_est[r], color=rater_colors[r], alpha=0.5, linewidth=1.5)

ax_r.set_xticks(range(1, R+1))
ax_r.set_xticklabels([f'R{r+1}' for r in range(R)], fontsize=9)
ax_r.set_ylabel('Threshold Value (logit)', fontsize=9); ax_r.set_ylim(-2, 2)
ax_r.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax_r.set_title('Rater Threshold Profiles', fontsize=11)
for k in range(K-1):
    ax_r.scatter([], [], marker=thresh_marks[k], color='gray', s=60, label=f'$\\phi_{{r,{k+1}}}$')
ax_r.legend(fontsize=8)

fig.suptitle('Wright Map — Hierarchical Rater Model (HRM)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'wright_map_hrm.png'), dpi=120, bbox_inches='tight')
plt.show()

NameError: name 'plt' is not defined

### 5b. Category Response Curves (CRC) — Rater-Specific Threshold Effects

**Interpretation**: The CRC for the HRM shows how different raters' threshold profiles translate into different scoring patterns. Raters with tightly clustered thresholds (central tendency) will have flatter, more overlap between middle categories. Raters with well-spread thresholds show cleaner category separation. The figure shows one item across all five raters — the shifting and reshaping of the CRC encapsulates each rater's unique scoring behaviour.

In [7]:
theta_range = np.linspace(-4, 4, 300)
cat_colors  = ['#3498DB', '#E67E22', '#2ECC71', '#9B59B6']

fig, axes = plt.subplots(1, R, figsize=(16, 4), sharey=True)
item_idx  = 9
for r in range(R):
    ax = axes[r]
    for k in range(K):
        probs = [hrm_probs(t, b_est[item_idx], phi_est[r])[k] for t in theta_range]
        ax.plot(theta_range, probs, color=cat_colors[k], linewidth=1.8, label=f'Cat {k}')
    thresh_str = ', '.join([f'{phi_est[r,k]:.2f}' for k in range(K-1)])
    ax.set_title(f'Rater {r+1}\n$\\phi$=[{thresh_str}]', fontsize=7)
    ax.set_xlim(-4, 4); ax.set_ylim(0, 1)
    ax.set_xlabel('$\\theta$', fontsize=9)
    if r == 0: ax.set_ylabel('Probability', fontsize=9)
    if r == 0: ax.legend(fontsize=8)

fig.suptitle(f'CRC — Item 10 across 5 Raters (HRM)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'crc_hrm.png'), dpi=120, bbox_inches='tight')
plt.show()

NameError: name 'np' is not defined

### 5c. Test Characteristic Curve (TCC)

**Interpretation**: The TCC for HRM shows that rater-induced scoring differences are not simply horizontal shifts (as in MFRM) but can change **shape** — the TCC slope may differ between raters if their threshold spacings differ. A rater with compressed thresholds (central tendency bias) produces a shallower TCC slope, meaning the test discriminates poorly between ability levels for that rater. This is a key diagnostic advantage of HRM over MFRM.

In [8]:
fig, ax = plt.subplots(figsize=(10, 6))
tcc_all = []
for r in range(R):
    tcc_r = np.zeros(len(theta_range))
    for i in range(I):
        for t_idx, t in enumerate(theta_range):
            pr = hrm_probs(t, b_est[i], phi_est[r])
            tcc_r[t_idx] += np.dot(np.arange(K), pr)
    tcc_all.append(tcc_r)
    thresh_str = f'{phi_est[r].mean():.2f}'
    ax.plot(theta_range, tcc_r, color=rater_colors[r],
            linewidth=2, label=f'Rater {r+1} (mean $\\phi$={thresh_str})')

tcc_arr = np.array(tcc_all)
ax.fill_between(theta_range, tcc_arr.min(axis=0), tcc_arr.max(axis=0),
                alpha=0.15, color='gray', label='Rater range')
ax.set_xlabel('$\\theta$ — Person Ability (logit)', fontsize=12)
ax.set_ylabel('Expected Total Score', fontsize=12)
ax.set_title('TCC by Rater — Hierarchical Rater Model (HRM)', fontsize=14)
ax.set_xlim(-4, 4); ax.set_ylim(0, I * (K - 1))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'tcc_hrm.png'), dpi=120, bbox_inches='tight')
plt.show()

NameError: name 'plt' is not defined

In [ ]:
# ── Posterior Parameter Density (Logit Scale) ─────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('HRM — Estimated Parameter Distributions (Logit Scale)', fontsize=13, fontweight='bold')

phi_flat = phi_est.ravel()

panels = [
    (axes[0], theta_est, r'$\hat{\theta}_j$  (person ability)',       'steelblue'),
    (axes[1], b_est,     r'$\hat{b}_i$  (item difficulty)',             'firebrick'),
    (axes[2], phi_flat,  r'$\hat{\phi}_{rk}$  (rater thresholds, flat)','mediumpurple'),
]

for ax, vals, title, color in panels:
    ax.hist(vals, bins=max(8, len(vals)//3), density=True,
            color=color, alpha=0.35, edgecolor='white')
    if len(vals) >= 3:
        xs = np.linspace(vals.min() - 0.5, vals.max() + 0.5, 300)
        kde = gaussian_kde(vals, bw_method='scott')
        ax.plot(xs, kde(xs), color=color, linewidth=2)
    ax.axvline(vals.mean(), color='black', linestyle='--', linewidth=1.2,
               label=f'mean={vals.mean():.2f}')
    ax.axvline(0, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.set_xlabel('Logit', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'{title}  (n={len(vals)})', fontsize=11)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(tmpdir, 'density_hrm.png'), dpi=120, bbox_inches='tight')
plt.show()
for name, vals in [('theta', theta_est), ('b', b_est), ('phi (flat)', phi_flat)]:
    print(f"{name:12s}: mean={vals.mean():.3f}  SD={vals.std():.3f}  "
          f"range=[{vals.min():.2f}, {vals.max():.2f}]")
